In [ ]:
import os
from google.colab import files

# Create folder
folder_path = "/content/pdfs"
os.makedirs(folder_path, exist_ok=True)

# Upload files
uploaded = files.upload()

# Save files into folder
for filename in uploaded.keys():
    with open(os.path.join(folder_path, filename), "wb") as f:
        f.write(uploaded[filename])

print("Files saved in:", folder_path)
print("Files:", os.listdir(folder_path))

Saving महाविकास आघाडी म्हणजे जनतेची फसवणूक_थिंक बॅंक_विनोद शिरसाठ पुस्तक.pdf to महाविकास आघाडी म्हणजे जनतेची फसवणूक_थिंक बॅंक_विनोद शिरसाठ पुस्तक (1).pdf
Saving मोदी की गांधी_थिंक बॅंक_विनोद शिरसाठ पुस्तक (1).pdf to मोदी की गांधी_थिंक बॅंक_विनोद शिरसाठ पुस्तक (1) (1).pdf
Saving स्थानिक पक्षांची रणनिती काय_थिंक बॅंक_विनोद शिरसाठ पुस्तक.pdf to स्थानिक पक्षांची रणनिती काय_थिंक बॅंक_विनोद शिरसाठ पुस्तक (1).pdf
Saving Family Business कसा वाढवावा_भाग्यश्री.pdf to Family Business कसा वाढवावा_भाग्यश्री (1).pdf
Saving एआय मध्ये करिअर करायचंय_भाग्यश्री.pdf to एआय मध्ये करिअर करायचंय_भाग्यश्री (1).pdf
Saving टॅक्स भरण्याची लोकांची तयारी नाही_भाग्यश्री.pdf to टॅक्स भरण्याची लोकांची तयारी नाही_भाग्यश्री (1).pdf
Saving डीप सीक एआय_भाग्यश्री.pdf to डीप सीक एआय_भाग्यश्री (1).pdf
Saving तुमचं आयुष्य सोपं करण्यासाठी_भाग्यश्री.pdf to तुमचं आयुष्य सोपं करण्यासाठी_भाग्यश्री (1).pdf
Saving तुमचं शहर नक्की कुणाच्या ताब्यात_भाग्यश्री.pdf to तुमचं शहर नक्की कुणाच्या ताब्यात_भाग्यश्री (1).pdf
Saving तुमचे निर्ण

In [ ]:
print(folder_path)

/content/pdfs


In [ ]:
!pip install -q -U google-genai pdfplumber gradio langchain-text-splitters langchain-community faiss-cpu sentence-transformers google-generativeai

In [ ]:
import os
import re
import glob
import gradio as gr
import pdfplumber
import google.generativeai as genai

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
MY_API_KEY = "API KEY"
MODEL_NAME = 'gemini-2.5-flash-lite'


DATASET_FOLDER = "/content/pdfs"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


vector_db = None
gemini_model = genai.GenerativeModel(MODEL_NAME)

/tmp/ipykernel_24605/2846393720.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or d

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def clean_marathi_text(text):
    if not text: return ""

    text = re.sub(r'न{2,}', '', text)

    text = re.sub(r'[^\u0900-\u097F\s0-9.,?!:;।\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def ingest_directory():
    global vector_db

    if not os.path.exists(DATASET_FOLDER):
        os.makedirs(DATASET_FOLDER)
        return f"❌ त्रुटी: '{DATASET_FOLDER}' फोल्डर सापडले नाही. कृपया कोलबमध्ये हे फोल्डर तयार करा."

    pdf_files = glob.glob(os.path.join(DATASET_FOLDER, "*.pdf"))

    if not pdf_files:
        return " फोल्डरमध्ये एकही PDF फाईल सापडली नाही. कृपया डाव्या बाजूला फाईल्स अपलोड करा."

    all_docs = []

    for file_path in pdf_files:
        file_name = os.path.basename(file_path)
        try:
            with pdfplumber.open(file_path) as pdf:
                full_text = ""
                for page in pdf.pages:
                    content = page.extract_text()
                    if content: full_text += content + "\n"

                cleaned = clean_marathi_text(full_text)


                text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
                chunks = text_splitter.split_text(cleaned)

                for chunk in chunks:
                    all_docs.append(Document(page_content=chunk, metadata={"source": file_name}))
        except Exception as e:
            print(f"⚠️ फाईल {file_name} वाचताना त्रुटी: {e}")

    if all_docs:
        vector_db = FAISS.from_documents(all_docs, embeddings)
        return f" यशस्वी! {len(pdf_files)} फाईल्स लोड झाल्या असून {len(all_docs)} माहितीचे तुकडे तयार झाले आहेत."

    return " कोणत्याही फाईलमधून मजकूर वाचता आला नाही."

In [ ]:
def ask_question(question, history):
    global vector_db, MY_API_KEY, gemini_model

    if vector_db is None:
        return "कृपया आधी 'डेटा लोड करा' या बटणावर क्लिक करा."

    try:

        genai.configure(api_key=MY_API_KEY)


        related_docs = vector_db.similarity_search(question, k=5)


        context_text = "\n\n".join([doc.page_content for doc in related_docs])
        sources = list(set([doc.metadata['source'] for doc in related_docs]))


        instruction = f"""
        तू एक मराठी तज्ज्ञ आहेस. तुला दिलेल्या माहितीच्या आधारे प्रश्नाचे उत्तर द्यायचे आहे.

        सूचना:
        १. उत्तर फक्त दिलेल्या संदर्भातून (Context) असावे.
        २. उत्तर मुद्देसूद आणि स्पष्ट मराठीत लिहा.
        ३. माहिती नसल्यास 'फाईलमध्ये उत्तर सापडले नाही' असे सांगा.

        संदर्भ (Context):
        {context_text}

        प्रश्न: {question}
        """

        response = gemini_model.generate_content(
            contents=instruction
        )

        answer = clean_marathi_text(response.text)
        source_info = f"\n\n📍 **संदर्भ फाईल:** {', '.join(sources)}"

        return answer + source_info

    except Exception as e:
        return f"माफ करा, काहीतरी तांत्रिक अडचण आली: {str(e)}"

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏆 मल्टी-ट्रान्सक्रिप्ट मराठी चॅटबॉट")
    gr.Markdown(f"डाव्या बाजूला असलेल्या `{DATASET_FOLDER}` फोल्डरमधील सर्व माहितीवर हा चॅटबॉट आधारित आहे.")

    with gr.Row():
        ingest_btn = gr.Button("🔄 डेटा लोड / रिफ्रेश करा (Refresh Dataset)", variant="primary")
        status_msg = gr.Textbox(label="सिस्टिम स्टेटस", interactive=False)

    ingest_btn.click(ingest_directory, outputs=status_msg)

    chat_ui = gr.ChatInterface(
        fn=ask_question,
        title="मराठीत संवाद साधा",
        description="तुमच्या ट्रान्सक्रिप्ट्समधील माहिती शोधण्यासाठी प्रश्न विचारा."
    )

demo.launch(debug=True)

/tmp/ipykernel_24605/830304479.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b96d6ec4319e55f931.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1468.63ms
